# Level 4 — 심층 분석: Grad-CAM 및 효율성 Trade-off

**목표**: 모델의 *동작 원리* 를 설명하고, FPS와 정확도의 trade-off 를 정량화합니다.

리포트 필수 산출물:
1. **속성별 정규화 Confusion Matrix 3개** — best 모델 기준.
2. **Grad-CAM 패널** — 같은 이미지에 대해 3개 head 가 각각 어디를 보는지 시각화 (예: `rainy + night + city street` 인 이미지).
3. 모든 백본에 대한 **FPS vs Avg-MF1 Pareto plot**.

본 노트북은 학습 노트북이 아니라 분석 노트북이지만, wandb 가 활성화되어 있으면 confusion matrix 이미지·Grad-CAM 패널·FPS 표를 같은 프로젝트의 별도 Run 으로 업로드합니다.

In [1]:
import os
import sys

repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/jmk05764/2026-HYU-AUE8088-PA2.git
else:
    !git -C /content/{repo_name} fetch origin
    !git -C /content/{repo_name} reset --hard origin/main
    !git -C /content/{repo_name} log --oneline -1

%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

!pip install -q -r requirements.txt

# Google Drive 마운트 — Level 3 체크포인트 로드에 필요
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/aue8088-pa2/checkpoints"

Mounted at /content/drive


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed
from src.utils.transforms import eval_transform
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, average_macro_f1, CLASS_NAMES
from src.utils.efficiency import measure_fps
from src.utils.wandb_logger import WandbLogger
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.xai.gradcam import GradCAM
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16
from src.models.vit import vit_small_patch16_224

set_seed(42, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
logger = WandbLogger(project=WANDB_PROJECT, run_name="level4-analysis", tags=["level4", "analysis"])

In [ ]:
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."
DATA_ROOT  = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)
    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")

val_ds     = BDDAttrDataset("../data/set_a", "val", transform=eval_transform())
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

# best 모델 (Level 3 Focal+Sampler+Mix ViT pretrained) — Drive에서 로드
import urllib.request

DEIT_URL  = "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth"
DEIT_PATH = "../checkpoints/deit_small_patch16_224.pth"
L3_CKPT   = f"{DRIVE_CKPT_DIR}/level3_focal+sampler+Mix.pth"

os.makedirs("../checkpoints", exist_ok=True)
if not os.path.exists(DEIT_PATH):
    print("DeiT weight 다운로드 중...")
    urllib.request.urlretrieve(DEIT_URL, DEIT_PATH)

def load_vit_pretrained(ckpt_path):
    m = vit_small_patch16_224().to(device)
    pre = torch.load(DEIT_PATH, map_location=device)["model"]
    def remap(sd):
        new = {}
        for k, v in sd.items():
            if k.startswith("head."): continue
            k = k.replace(".mlp.fc1.", ".mlp.0.").replace(".mlp.fc2.", ".mlp.3.")
            new[k] = v
        return new
    m.load_state_dict(remap(pre), strict=False)
    ckpt = torch.load(ckpt_path, map_location=device)
    m.load_state_dict(ckpt["state_dict"])
    return m

model = load_vit_pretrained(L3_CKPT)
print("best model loaded (ViT-S pretrained, Level 3 Focal+Sampler+Mix)")

In [ ]:
# 속성별 정규화 Confusion Matrix 생성 및 시각화
preds, probs, targets, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(preds, targets, normalize="true")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", ax=ax, cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a])
    ax.set_title(f"{a}")
    ax.set_xlabel("예측 (pred)"); ax.set_ylabel("정답 (true)")
fig.tight_layout()
logger.log_image("analysis/confusion_matrices", fig)

# 속성별 개별 confusion matrix 도 분리해서 업로드
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"analysis/cm_{a}", cms[a], CLASS_NAMES[a])
plt.show()

In [ ]:
# Grad-CAM — val_ds.samples에서 특정 레이블 조합 이미지를 직접 선택.
# blocks[-1] 출력은 norm→x[:,0](CLS) 경로로 패치 gradient=0이므로 blocks[-2] 사용.

# (weather, scene, timeofday) 인덱스 조합
# weather:   0=clear, 1=overcast, 2=rainy, 3=snowy, 4=foggy, 5=partly cloudy
# scene:     0=city street, 1=highway, 2=residential
# timeofday: 0=daytime, 1=night, 2=dawn/dusk
COMBOS = [
    (2, 0, 1),  # rainy + city street + night  (리포트 예시)
    (2, 0, 0),  # rainy + city street + daytime
    (0, 1, 0),  # clear + highway + daytime
    (4, 0, 0),  # foggy + city street + daytime
    (0, 0, 1),  # clear + city street + night
    (1, 2, 2),  # overcast + residential + dawn/dusk
]

combo_indices = []
for target in COMBOS:
    for i, s in enumerate(val_ds.samples):
        if (s.weather, s.scene, s.timeofday) == target:
            combo_indices.append(i)
            break

combo_indices = combo_indices[:6]
n_imgs = len(combo_indices)
x_sel = torch.stack([val_ds[i]["image"] for i in combo_indices]).to(device)
col_labels = [
    f"W:{CLASS_NAMES['weather'][val_ds.samples[i].weather]}\n"
    f"S:{CLASS_NAMES['scene'][val_ds.samples[i].scene]}\n"
    f"T:{CLASS_NAMES['timeofday'][val_ds.samples[i].timeofday]}"
    for i in combo_indices
]

target_layer = model.blocks[-2]
gc = GradCAM(model, target_layer)

fig, axes = plt.subplots(3, n_imgs, figsize=(3 * n_imgs, 9))
fig.suptitle("Grad-CAM — 속성별 head가 주목하는 영역 (ViT-S pretrained)", fontsize=13)
for row, attr in enumerate(ATTRIBUTES):
    cam = gc(x_sel, lambda out, a=attr: out[a].max(dim=-1).values.sum())
    for col in range(n_imgs):
        img = x_sel[col].cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())
        axes[row, col].imshow(img)
        axes[row, col].imshow(cam[col].cpu().numpy(), cmap="jet", alpha=0.45)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(col_labels[col], fontsize=8)
        if col == 0:
            axes[row, col].set_ylabel(attr, fontsize=12)

fig.tight_layout()
logger.log_image("analysis/gradcam_panel", fig)
plt.show()

In [ ]:
# TODO 3: FPS 측정 + Pareto Plot
# Level 1~3에서 얻은 Avg-MF1 수치를 기입하세요.
BACKBONE_INFO = [
    # (model_fn,            name,        avg_mf1)
    (VGG16,                 "VGG-16",    0.562),
    (resnet18,              "ResNet-18", 0.658),
    (resnet50,              "ResNet-50", 0.633),
    (vit_small_patch16_224, "ViT-S/16",  0.683),  # Level 2 pretrained
]

fps_rows = []
for model_fn, name, mf1 in BACKBONE_INFO:
    m = model_fn().to(device).eval()
    fps = measure_fps(m, device, batch_size=1, n_warmup=20, n_iter=200)
    print(f"{name:12s} FPS={fps:7.1f}  Avg-MF1={mf1:.3f}")
    fps_rows.append([name, round(fps, 1), mf1])
    del m

logger.log_table("analysis/fps", ["backbone", "FPS", "Avg-MF1"], fps_rows)

# Pareto Plot
names  = [r[0] for r in fps_rows]
fps_v  = [r[1] for r in fps_rows]
mf1_v  = [r[2] for r in fps_rows]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(fps_v, mf1_v, s=120, zorder=3)
for name, x, y in zip(names, fps_v, mf1_v):
    ax.annotate(name, (x, y), textcoords="offset points", xytext=(8, 4), fontsize=10)

# Pareto front (FPS 오름차순 정렬 후 MF1 단조증가 지점 연결)
sorted_pts = sorted(zip(fps_v, mf1_v))
pareto, best_mf1 = [], -1
for fx, fy in sorted_pts:
    if fy > best_mf1:
        pareto.append((fx, fy))
        best_mf1 = fy
if len(pareto) > 1:
    px, py = zip(*pareto)
    ax.plot(px, py, "r--", linewidth=1.5, label="Pareto front")

ax.set_xlabel("FPS (batch=1, T4 GPU)", fontsize=12)
ax.set_ylabel("Avg Macro-F1", fontsize=12)
ax.set_title("FPS vs Avg-Macro-F1 Trade-off", fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
logger.log_image("analysis/pareto", fig)
plt.show()

In [ ]:
logger.finish()